In [1]:
import joblib
import numpy as np
import pandas as pd
import gc
import time
from pathlib import Path
from sklearn.metrics import confusion_matrix

# ============================================================
# CONFIG
# ============================================================
LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
PIXEL_AREA_M2 = 100  # 10m × 10m
CHUNK_SIZE = 500_000

In [2]:
# ============================================================
# CELL 1: LOAD ARTIFACTS
# ============================================================
print('Loading saved MLP, scaler, and feature column list...')

mlp = joblib.load(LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_best.joblib')
scaler = joblib.load(LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_scaler.joblib')

with open(LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_feature_cols.txt') as f:
    FEATURE_COLS = [line.strip() for line in f if line.strip()]

print(f'  MLP loaded: {mlp.hidden_layer_sizes} hidden layers')
print(f'  Scaler loaded: {scaler.mean_.shape[0]} features')
print(f'  Feature columns: {len(FEATURE_COLS)} features')
assert len(FEATURE_COLS) == scaler.mean_.shape[0], 'Feature count mismatch!'



Loading saved MLP, scaler, and feature column list...
  MLP loaded: (512, 256, 128) hidden layers
  Scaler loaded: 66 features
  Feature columns: 66 features


In [3]:
# ============================================================
# CELL 2: LOAD FULL PERU-WIDE DATA
# ============================================================
print('\nLoading Peru-wide data...')
t0 = time.time()

dfs = []
for region in REGIONS:
    df = pd.read_parquet(LOCAL_DIR / f'Merged\\{region.lower()}_combined.parquet')
    df['region'] = region
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(df_all):,} pixels in {time.time()-t0:.1f}s')
print(f'Memory: {df_all.memory_usage(deep=True).sum() / 1e9:.2f} GB')

# Sanity check that all expected feature columns exist in df_all
missing = [c for c in FEATURE_COLS if c not in df_all.columns]
if missing:
    raise ValueError(f'Missing columns in df_all: {missing}')
print(f'All {len(FEATURE_COLS)} feature columns present in df_all')




Loading Peru-wide data...
Loaded 15,242,639 pixels in 27.7s
Memory: 5.30 GB
All 66 feature columns present in df_all


In [4]:
# ============================================================
# CELL 3: EXTRACT FEATURES, STANDARDISE, PREDICT IN CHUNKS
# ============================================================
print('\nPredicting melt probabilities for all 15.2M pixels...')

# Allocate output array
n_total = len(df_all)
probs = np.zeros(n_total, dtype=np.float32)

t0 = time.time()
for i in range(0, n_total, CHUNK_SIZE):
    end = min(i + CHUNK_SIZE, n_total)
    
    # Extract this chunk's features in the saved column order
    X_chunk = df_all[FEATURE_COLS].iloc[i:end].values
    
    # Apply the saved scaler
    X_chunk_scaled = scaler.transform(X_chunk)
    
    # Predict melt probability (class 1)
    probs[i:end] = mlp.predict_proba(X_chunk_scaled)[:, 1]
    
    del X_chunk, X_chunk_scaled
    
    if (i // CHUNK_SIZE) % 5 == 0 or end == n_total:
        print(f'  {end:,} / {n_total:,} done '
              f'({(end/n_total)*100:.0f}%, '
              f'elapsed {time.time()-t0:.0f}s)')

gc.collect()
df_all['prob_mlp'] = probs
print(f'Prediction done in {time.time()-t0:.1f}s')
print(f'Probability stats: mean={probs.mean():.3f}, '
      f'median={np.median(probs):.3f}, '
      f'std={probs.std():.3f}')




Predicting melt probabilities for all 15.2M pixels...
  500,000 / 15,242,639 done (3%, elapsed 57s)
  3,000,000 / 15,242,639 done (20%, elapsed 131s)
  5,500,000 / 15,242,639 done (36%, elapsed 201s)
  8,000,000 / 15,242,639 done (52%, elapsed 276s)
  10,500,000 / 15,242,639 done (69%, elapsed 344s)
  13,000,000 / 15,242,639 done (85%, elapsed 409s)
  15,242,639 / 15,242,639 done (100%, elapsed 471s)
Prediction done in 473.1s
Probability stats: mean=0.246, median=0.002, std=0.376


In [5]:
# ============================================================
# CELL 4: THRESHOLDED SPATIAL OVERLAP
# ============================================================
print('\nComputing thresholded spatial overlap...')

# Get actual melt pixel count and area
n_actual_melt = (df_all['melt_label'] == 1).sum()
actual_melt_area_km2 = n_actual_melt * PIXEL_AREA_M2 / 1e6

print(f'Actual melt pixels: {n_actual_melt:,}')
print(f'Actual melt area:   {actual_melt_area_km2:.1f} km²')

def thresholded_overlap(probs, labels, target_n_melt):
    """Threshold predictions to predict exactly target_n_melt pixels as melt.
    Returns a dict of overlap metrics."""
    # Identify top-N most-vulnerable pixels via argsort (robust to ties)
    top_indices = np.argsort(probs)[::-1][:target_n_melt]
    predicted_melt = np.zeros(len(probs), dtype=bool)
    predicted_melt[top_indices] = True
    
    threshold = probs[top_indices[-1]]
    
    actual_melt = labels == 1
    n_predicted = predicted_melt.sum()
    n_actual = actual_melt.sum()
    n_intersect = (predicted_melt & actual_melt).sum()
    n_union = (predicted_melt | actual_melt).sum()
    
    return {
        'threshold': threshold,
        'n_predicted': n_predicted,
        'n_actual': n_actual,
        'n_intersect': n_intersect,
        'overlap_pct': n_intersect / n_actual * 100,
        'precision_pct': n_intersect / n_predicted * 100,
        'iou': n_intersect / n_union,
    }

result_mlp = thresholded_overlap(
    df_all['prob_mlp'].values,
    df_all['melt_label'].values,
    target_n_melt=n_actual_melt
)

print('\n=== MLP Spatial Overlap Result ===')
for k, v in result_mlp.items():
    if isinstance(v, float):
        print(f'  {k:18s} {v:.4f}')
    else:
        print(f'  {k:18s} {v:,}')




Computing thresholded spatial overlap...
Actual melt pixels: 2,971,684
Actual melt area:   297.2 km²

=== MLP Spatial Overlap Result ===
  threshold          0.73723965883255
  n_predicted        2,971,684
  n_actual           2,971,684
  n_intersect        2,092,237
  overlap_pct        70.4058
  precision_pct      70.4058
  iou                0.5433


In [6]:
# ============================================================
# CELL 5: COMPARISON WITH PREVIOUS RF RESULTS
# ============================================================
print('\n=== COMPARISON WITH PREVIOUS RESULTS ===')
print(f'{"Model":<30} {"Overlap %":<12} {"IoU":<8}')
print(f'{"RF EASD-only (strict)":<30} {"54.4":<12} {"0.374":<8}')
print(f'{"RF EASD+PC10 (strict)":<30} {"68.1":<12} {"0.516":<8}')
print(f'{"RF EASD-only (loose)":<30} {"62.7":<12} {"0.457":<8}')
print(f'{"RF EASD+PC10 (loose)":<30} {"74.4":<12} {"0.592":<8}')
print(f'{"MLP (66 features, AE+E+D)":<30} '
      f'{result_mlp["overlap_pct"]:<12.2f} {result_mlp["iou"]:<8.4f}')
print(f'\nReference: Darina (2024) reported 74.9% spatial overlap')




=== COMPARISON WITH PREVIOUS RESULTS ===
Model                          Overlap %    IoU     
RF EASD-only (strict)          54.4         0.374   
RF EASD+PC10 (strict)          68.1         0.516   
RF EASD-only (loose)           62.7         0.457   
RF EASD+PC10 (loose)           74.4         0.592   
MLP (66 features, AE+E+D)      70.41        0.5433  

Reference: Darina (2024) reported 74.9% spatial overlap


In [7]:
# ============================================================
# CELL 6: SAVE PREDICTIONS
# ============================================================
output_cols = ['region', 'lon', 'lat', 'melt_label', 'edge_distance',
               'elevation', 'prob_mlp']
df_predictions = df_all[output_cols]
df_predictions.to_parquet(LOCAL_DIR / 'Predictions\\predictions_mlp.parquet')
print(f'\nSaved MLP predictions: {len(df_predictions):,} rows '
      f'to predictions_mlp.parquet')


Saved MLP predictions: 15,242,639 rows to predictions_mlp.parquet


In [9]:
# Quickly testing MLP with looser stratification to see if it can beat the 74.4% overlap (RF, EASD+PC10)
import joblib
import numpy as np
import pandas as pd
import gc
import time
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix

LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
RANDOM_STATE = 42
PER_CATEGORY_TARGET = 100000  # Darina-style looser cap

# Load data
print('Loading data...')
dfs = [pd.read_parquet(LOCAL_DIR / f'Merged\\{r.lower()}_combined.parquet') for r in REGIONS]
df_all = pd.concat(dfs, ignore_index=True)
ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]

# 5-quintile stratification with LOOSER 100k cap per category
df_all['quintile'] = pd.qcut(df_all['edge_distance'], q=5, labels=False) + 1

sampled = []
for (q, m), group in df_all.groupby(['quintile', 'melt_label']):
    n_take = min(PER_CATEGORY_TARGET, len(group))
    sampled.append(group.sample(n=n_take, random_state=RANDOM_STATE))
df_train_full = pd.concat(sampled, ignore_index=True)

print(f'Training set: {len(df_train_full):,} samples '
      f'({df_train_full["melt_label"].mean()*100:.1f}% melt)')
print('Per (bin × class):')
print(df_train_full.groupby(['quintile', 'melt_label']).size().unstack(fill_value=0))

# Features
FEATURE_COLS = ['elevation', 'edge_distance'] + ae_cols

# Train/val split
X = df_train_full[FEATURE_COLS].values
y = df_train_full['melt_label'].values
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# Scaler
scaler_loose = StandardScaler()
X_train_scaled = scaler_loose.fit_transform(X_train)
X_val_scaled = scaler_loose.transform(X_val)
joblib.dump(scaler_loose, LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_scaler_loose.joblib')

# Train MLP with same hyperparameters as your locked best
print('\nTraining MLP with looser stratification...')
t0 = time.time()
mlp_loose = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    alpha=0.0001,
    batch_size=512,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
    verbose=False
)
mlp_loose.fit(X_train_scaled, y_train)
print(f'Trained in {time.time()-t0:.1f}s ({mlp_loose.n_iter_} epochs)')

# Val performance for sanity check
y_pred = mlp_loose.predict(X_val_scaled)
cm = confusion_matrix(y_val, y_pred)
ice_err = 1 - cm[0,0]/cm[0].sum()
melt_err = 1 - cm[1,1]/cm[1].sum()
avg_err = 1 - (cm[0,0] + cm[1,1])/cm.sum()
print(f'\nValidation: Ice {ice_err:.4f}, Melt {melt_err:.4f}, Avg {avg_err:.4f}')

# Save
joblib.dump(mlp_loose, LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_loose.joblib')
with open(LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_feature_cols.txt') as f:
    pass  # already saved, same features
print('Saved mlp_loose.joblib')

# Now run spatial overlap (using your existing pipeline pattern)
del X, X_train, X_val, X_train_scaled, X_val_scaled, df_train_full
gc.collect()

# Predict on full Peru-wide
print('\nPredicting on full Peru-wide population...')
probs = np.zeros(len(df_all), dtype=np.float32)
CHUNK_SIZE = 500_000
for i in range(0, len(df_all), CHUNK_SIZE):
    end = min(i + CHUNK_SIZE, len(df_all))
    X_chunk = df_all[FEATURE_COLS].iloc[i:end].values
    X_chunk_scaled = scaler_loose.transform(X_chunk)
    probs[i:end] = mlp_loose.predict_proba(X_chunk_scaled)[:, 1]
    del X_chunk, X_chunk_scaled
    if (i // CHUNK_SIZE) % 5 == 0 or end == len(df_all):
        print(f'  {end:,} / {len(df_all):,}')

# Spatial overlap
n_actual_melt = (df_all['melt_label'] == 1).sum()
top_indices = np.argsort(probs)[::-1][:n_actual_melt]
predicted_melt = np.zeros(len(probs), dtype=bool)
predicted_melt[top_indices] = True
actual_melt = df_all['melt_label'].values == 1
n_intersect = (predicted_melt & actual_melt).sum()
overlap_pct = n_intersect / n_actual_melt * 100
iou = n_intersect / (predicted_melt | actual_melt).sum()

print(f'\n=== MLP (looser stratification) Spatial Overlap ===')
print(f'  Overlap: {overlap_pct:.2f}%')
print(f'  IoU:     {iou:.4f}')
print(f'\nCompare to:')
print(f'  MLP strict:       70.4%')
print(f'  RF loose (EASD+PC10): 74.4%')
print(f'  Darina 2024:      74.9%')

Loading data...
Training set: 845,174 samples (40.8% melt)
Per (bin × class):
melt_label       0       1
quintile                  
1           100000  100000
2           100000  100000
3           100000  100000
4           100000   37983
5           100000    7191

Training MLP with looser stratification...
Trained in 765.3s (33 epochs)

Validation: Ice 0.0977, Melt 0.1268, Avg 0.1095
Saved mlp_loose.joblib

Predicting on full Peru-wide population...
  500,000 / 15,242,639
  3,000,000 / 15,242,639
  5,500,000 / 15,242,639
  8,000,000 / 15,242,639
  10,500,000 / 15,242,639
  13,000,000 / 15,242,639
  15,242,639 / 15,242,639

=== MLP (looser stratification) Spatial Overlap ===
  Overlap: 76.98%
  IoU:     0.6257

Compare to:
  MLP strict:       70.4%
  RF loose (EASD+PC10): 74.4%
  Darina 2024:      74.9%


In [10]:
# ============================================================
# TRAINING PERFORMANCE / OVERFIT GAP
# df_train_full was deleted, so we reconstruct it identically
# from df_all (still in memory) using the same random_state.
# ============================================================
sampled_recon = []
for (q, m), group in df_all.groupby(['quintile', 'melt_label']):
    n_take = min(PER_CATEGORY_TARGET, len(group))
    sampled_recon.append(group.sample(n=n_take, random_state=RANDOM_STATE))
df_train_recon = pd.concat(sampled_recon, ignore_index=True)
del sampled_recon

X_tr_recon = scaler_loose.transform(df_train_recon[FEATURE_COLS].values)
y_tr_recon = df_train_recon['melt_label'].values
del df_train_recon

tr_pred = mlp_loose.predict(X_tr_recon)
del X_tr_recon
gc.collect()

cm_tr   = confusion_matrix(y_tr_recon, tr_pred)
tr_ice  = 1 - cm_tr[0, 0] / cm_tr[0].sum()
tr_melt = 1 - cm_tr[1, 1] / cm_tr[1].sum()
tr_avg  = 1 - (cm_tr[0, 0] + cm_tr[1, 1]) / cm_tr.sum()

print('=== Train vs Val Performance (mlp_loose) ===')
print(f'{"Metric":<12} {"Train":>8} {"Val":>8} {"Gap (val-train)":>16}')
print('-' * 46)
print(f'{"Ice error":<12} {tr_ice:>8.4f} {ice_err:>8.4f} {ice_err - tr_ice:>+16.4f}')
print(f'{"Melt error":<12} {tr_melt:>8.4f} {melt_err:>8.4f} {melt_err - tr_melt:>+16.4f}')
print(f'{"Avg error":<12} {tr_avg:>8.4f} {avg_err:>8.4f} {avg_err - tr_avg:>+16.4f}')

=== Train vs Val Performance (mlp_loose) ===
Metric          Train      Val  Gap (val-train)
----------------------------------------------
Ice error      0.0718   0.0977          +0.0259
Melt error     0.0928   0.1268          +0.0339
Avg error      0.0804   0.1095          +0.0292


In [13]:
import joblib
import numpy as np
import pandas as pd
import gc
import time
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix

LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
RANDOM_STATE = 42
N_PER_CLASS = 250_000  # 500k total, balanced

# Load
dfs = [pd.read_parquet(LOCAL_DIR / f'Merged\\{r.lower()}_combined.parquet') for r in REGIONS]
df_all = pd.concat(dfs, ignore_index=True)
ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]

# Simple class-balanced random sampling, no edge-distance stratification
melt_sample = df_all[df_all['melt_label'] == 1].sample(
    n=min(N_PER_CLASS, (df_all['melt_label'] == 1).sum()),
    random_state=RANDOM_STATE)
nonmelt_sample = df_all[df_all['melt_label'] == 0].sample(
    n=N_PER_CLASS, random_state=RANDOM_STATE)
df_train_full = pd.concat([melt_sample, nonmelt_sample], ignore_index=True)
print(f'Class-balanced sample: {len(df_train_full):,} pixels, '
      f'{df_train_full["melt_label"].mean()*100:.1f}% melt')

# Features: AE only + edge distance
FEATURE_COLS_AE = ae_cols + ['edge_distance']
print(f'Using {len(FEATURE_COLS_AE)} features (AE only + edge distance)')

X = df_train_full[FEATURE_COLS_AE].values
y = df_train_full['melt_label'].values
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler_ae = StandardScaler()
X_train_scaled = scaler_ae.fit_transform(X_train)
X_val_scaled = scaler_ae.transform(X_val)

# Train MLP
print('\nTraining AE-edge MLP...')
t0 = time.time()
mlp_ae = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    alpha=0.0001,
    batch_size=512,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
    verbose=False
)
mlp_ae.fit(X_train_scaled, y_train)
print(f'Trained in {time.time()-t0:.1f}s ({mlp_ae.n_iter_} epochs)')

y_pred = mlp_ae.predict(X_val_scaled)
cm = confusion_matrix(y_val, y_pred)
ice_err = 1 - cm[0,0]/cm[0].sum()
melt_err = 1 - cm[1,1]/cm[1].sum()
avg_err = 1 - (cm[0,0] + cm[1,1])/cm.sum()
print(f'Validation: Ice {ice_err:.4f}, Melt {melt_err:.4f}, Avg {avg_err:.4f}')

# Save
joblib.dump(mlp_ae, LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_ae_edge_no_stratification.joblib')
joblib.dump(scaler_ae, LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_scaler_ae_edge_no_stratification.joblib')

# Free memory before predicting
del X, X_train, X_val, X_train_scaled, X_val_scaled, df_train_full
gc.collect()

# Spatial overlap on Peru-wide
print('\nPredicting on full Peru-wide population...')
probs = np.zeros(len(df_all), dtype=np.float32)
CHUNK = 500_000
for i in range(0, len(df_all), CHUNK):
    end = min(i + CHUNK, len(df_all))
    X_chunk = df_all[FEATURE_COLS_AE].iloc[i:end].values
    X_chunk_scaled = scaler_ae.transform(X_chunk)
    probs[i:end] = mlp_ae.predict_proba(X_chunk_scaled)[:, 1]
    del X_chunk, X_chunk_scaled

n_actual_melt = (df_all['melt_label'] == 1).sum()
top_indices = np.argsort(probs)[::-1][:n_actual_melt]
predicted_melt = np.zeros(len(probs), dtype=bool)
predicted_melt[top_indices] = True
actual_melt = df_all['melt_label'].values == 1
n_intersect = (predicted_melt & actual_melt).sum()
overlap_pct = n_intersect / n_actual_melt * 100
iou = n_intersect / (predicted_melt | actual_melt).sum()

print(f'\n=== AE-edge MLP (no edge-distance stratification) ===')
print(f'  Overlap: {overlap_pct:.2f}%')
print(f'  IoU:     {iou:.4f}')

Class-balanced sample: 500,000 pixels, 50.0% melt
Using 65 features (AE only + edge distance)

Training AE-edge MLP...
Trained in 313.5s (29 epochs)
Validation: Ice 0.1050, Melt 0.0699, Avg 0.0874

Predicting on full Peru-wide population...

=== AE-edge MLP (no edge-distance stratification) ===
  Overlap: 80.58%
  IoU:     0.6747


In [14]:
# ============================================================
# CHECK OVERFITTING GAP: TRAIN vs VAL PERFORMANCE
# ============================================================
# Re-extract train and val arrays for evaluation
# (We deleted the originals to save memory after training)

print('Reconstructing train/val splits for overfitting check...')

# Re-sample using same random_state to get exactly the same train/val split
melt_sample = df_all[df_all['melt_label'] == 1].sample(
    n=min(N_PER_CLASS, (df_all['melt_label'] == 1).sum()),
    random_state=RANDOM_STATE)
nonmelt_sample = df_all[df_all['melt_label'] == 0].sample(
    n=N_PER_CLASS, random_state=RANDOM_STATE)
df_train_full = pd.concat([melt_sample, nonmelt_sample], ignore_index=True)

X = df_train_full[FEATURE_COLS_AE].values
y = df_train_full['melt_label'].values
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# Apply the same scaler we trained earlier
X_train_scaled = scaler_ae.transform(X_train)
X_val_scaled = scaler_ae.transform(X_val)

# === TRAIN performance ===
y_pred_train = mlp_ae.predict(X_train_scaled)
cm_train = confusion_matrix(y_train, y_pred_train)
ice_err_train = 1 - cm_train[0,0]/cm_train[0].sum()
melt_err_train = 1 - cm_train[1,1]/cm_train[1].sum()
avg_err_train = 1 - (cm_train[0,0] + cm_train[1,1])/cm_train.sum()

# === VAL performance (for reference) ===
y_pred_val = mlp_ae.predict(X_val_scaled)
cm_val = confusion_matrix(y_val, y_pred_val)
ice_err_val = 1 - cm_val[0,0]/cm_val[0].sum()
melt_err_val = 1 - cm_val[1,1]/cm_val[1].sum()
avg_err_val = 1 - (cm_val[0,0] + cm_val[1,1])/cm_val.sum()

# === Report ===
print('\n=== OVERFITTING ANALYSIS ===')
print(f'{"Metric":<12} {"Train":<10} {"Val":<10} {"Gap (val-train)":<15}')
print('-' * 50)
print(f'{"Ice error":<12} {ice_err_train:<10.4f} {ice_err_val:<10.4f} '
      f'{ice_err_val - ice_err_train:+.4f}')
print(f'{"Melt error":<12} {melt_err_train:<10.4f} {melt_err_val:<10.4f} '
      f'{melt_err_val - melt_err_train:+.4f}')
print(f'{"Avg error":<12} {avg_err_train:<10.4f} {avg_err_val:<10.4f} '
      f'{avg_err_val - avg_err_train:+.4f}')

print(f'\nEpochs trained: {mlp_ae.n_iter_}')
print(f'Training samples: {len(y_train):,}')
print(f'Validation samples: {len(y_val):,}')

# Free memory after we're done
del X, X_train, X_val, X_train_scaled, X_val_scaled, df_train_full
import gc; gc.collect()

Reconstructing train/val splits for overfitting check...

=== OVERFITTING ANALYSIS ===
Metric       Train      Val        Gap (val-train)
--------------------------------------------------
Ice error    0.0756     0.1050     +0.0294
Melt error   0.0423     0.0699     +0.0276
Avg error    0.0589     0.0874     +0.0285

Epochs trained: 29
Training samples: 400,000
Validation samples: 100,000


0